In [2]:
%pip install --quiet --upgrade s3fs

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.33.13 requires botocore==1.34.131, but you have botocore 1.36.3 which is incompatible.
boto3 1.34.131 requires botocore<1.35.0,>=1.34.131, but you have botocore 1.36.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from __future__ import print_function

%matplotlib inline

import sys
import zipfile
from dateutil.parser import parse
import json
from random import shuffle
import random
import datetime
import os

import boto3
import s3fs
import sagemaker
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Checkbox

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
# set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

In [3]:
sagemaker_session = sagemaker.Session()

In [4]:
s3_bucket = "tdsp-projects-dev"
s3_prefix = "mds/sales_incentive/data_eng/04_fea"
s3_output_prefix = "mds/sales_incentive/sandbox/ankit"
de_timestamp = "20241210070332"

role = sagemaker.get_execution_role()

In [5]:
region = sagemaker_session.boto_region_name

s3_data_path  = "s3://{}/{}/{}/develop/output_feature_pin_data.parquet/".format(s3_bucket, s3_prefix, de_timestamp)
s3_output_path = "s3://{}/{}/forecast_output/{}".format(s3_bucket, s3_output_prefix, de_timestamp)

In [6]:
image_name = sagemaker.image_uris.retrieve("forecasting-deepar", region)

In [7]:
data = pd.read_parquet(s3_data_path).fillna(0)

In [8]:
data['start'] = pd.to_datetime(data['year_month']).dt.strftime("%Y-%m-%d %H:%M:%S")

In [9]:
data = data.sort_values('year_month')

In [10]:
data.columns

Index(['model', 'model_year', 'year_month', 'toyota_region_area', 'nameplate',
       'body_style', 'drive_type', 'trim', 'transmission', 'cylinder',
       'toyota_subsegment', 'fuel_type', 'vehicle_price',
       'all_channel_incentive_spend_per_unit', 'total_sales_estimate',
       'cash_incentive_offer_per_unit', 'lease_incentive_offer_per_unit',
       'finance_incentive_offer_per_unit', 'type_of_sale',
       'channel_penetration_of_total_sales',
       'channel_incentive_offer_per_unit', 'channel_sales_estimate',
       'channel_total_incentive', 'cash_sales_estimate',
       'cash_total_incentive', 'finance_sales_estimate',
       'finance_total_incentive', 'lease_sales_estimate',
       'lease_total_incentive', 'channel_customer_facing_transaction_price',
       'channel_revenue_estimate', 'other_channel_sales_estimate',
       'other_channel_incentive_per_unit', 'model_division_name',
       'transaction_sales_count', 'wholesale_count', 'allocation_count',
       'freight_cou

In [11]:
data.head(10)

,model,model_year,year_month,toyota_region_area,nameplate,body_style,drive_type,trim,transmission,cylinder,...,pct_of_avg_3mon_ground_stock_count,switch_first_year_month,switch_last_year_month,switch_model_year,months_since_model_year_switch,first_year_month,last_year_month,current_model_year,months_since_model_year_superseded,start
162830,Avalon,2011,2012-01-01,Kansas City,Toyota,N/A,N/A,N/A,N/A,N/A,...,0.0,0,0,0,0.0,2012-01-01,2012-01-01,0,0.0,2012-01-01 00:00:00
150592,Avalon,2011,2012-01-01,Denver,Toyota,N/A,N/A,N/A,N/A,N/A,...,0.0,0,0,0,0.0,2012-01-01,2012-01-01,0,0.0,2012-01-01 00:00:00
185144,Prius,2011,2012-01-01,New York,Toyota,N/A,N/A,N/A,N/A,N/A,...,0.0,0,0,0,0.0,2012-01-01,2012-01-01,0,0.0,2012-01-01 00:00:00
90678,Highlander,2012,2012-01-01,Gulf States Toyota,Toyota,N/A,N/A,N/A,N/A,N/A,...,0.0,2012-01-01,2012-10-01,1,0.0,2012-01-01,2012-09-01,1,0.0,2012-01-01 00:00:00
24946,GS 450H,2011,2012-01-01,Central Area,Lexus,N/A,N/A,N/A,N/A,N/A,...,0.0,0,0,0,0.0,2012-01-01,2012-05-01,1,0.0,2012-01-01 00:00:00
28641,Highlander Hybrid,2012,2012-01-01,Gulf States Toyota,Toyota,N/A,N/A,N/A,N/A,N/A,...,0.0,2012-11-01,2012-09-01,0,0.0,2012-01-01,2012-09-01,1,0.0,2012-01-01 00:00:00
153120,RAV4,2011,2012-01-01,South East Toyota,Toyota,N/A,N/A,N/A,N/A,N/A,...,0.0,0,0,0,0.0,2012-01-01,2012-01-01,0,0.0,2012-01-01 00:00:00
123011,4Runner,2011,2012-01-01,Portland,Toyota,N/A,N/A,N/A,N/A,N/A,...,0.0,0,0,0,0.0,2012-01-01,2012-01-01,0,0.0,2012-01-01 00:00:00
114137,Prius v,2012,2012-01-01,South East Toyota,Toyota,N/A,N/A,N/A,N/A,N/A,...,0.0,0,0,0,0.0,2012-01-01,2012-10-01,1,0.0,2012-01-01 00:00:00
66766,Prius v,2012,2012-01-01,New York,Toyota,N/A,N/A,N/A,N/A,N/A,...,0.0,0,0,0,0.0,2012-01-01,2012-10-01,1,0.0,2012-01-01 00:00:00


In [12]:
import warnings
warnings.filterwarnings('ignore')
date_col = ['start']
target_col = ['channel_sales_estimate']
cat_cols = ['toyota_region_area','model','current_model_year','type_of_sale']
dynamic_cols = ['vehicle_price','cash_incentive_offer_per_unit','finance_incentive_offer_per_unit','lease_incentive_offer_per_unit',
                'wholesale_count','dealer_stock_count','allocation_count','freight_count',
                'months_since_model_year_superseded','months_since_model_year_switch']
all_cols = date_col+cat_cols+dynamic_cols+target_col

agg_functions = {'start':'min'}
cols_to_convert_timeseries = dynamic_cols+target_col
agg_functions.update({k:v for (k,v) in zip(cols_to_convert_timeseries, [lambda x: tuple(x.to_list())] * len(cols_to_convert_timeseries))})

# 1-month prediction and test. Change test_start specifies prediction month and train end
test_start = "2024-10-01 00:00:00"
data_train = data[data['start']<test_start].groupby(cat_cols).agg(agg_functions).reset_index()
train_models = data_train['model'].unique()


# data_test = data[data['model'].isin(train_models)].copy().groupby(cat_cols).agg(agg_functions).reset_index()
data_test = data_train.copy()

# Fill forward (copy) dynamic features from last training month
fill_forward_cols = ['vehicle_price','cash_incentive_offer_per_unit','finance_incentive_offer_per_unit','lease_incentive_offer_per_unit',
                'wholesale_count','dealer_stock_count','allocation_count','freight_count']
increment_forward_cols = ['months_since_model_year_superseded','months_since_model_year_switch']
for row in range(len(data_test)):
    for column in fill_forward_cols:
        data_test[column].iloc[row] += (data_test[column].iloc[row][-1],)
    for column in increment_forward_cols:
        data_test[column].iloc[row] += (data_test[column].iloc[row][-1]+1,)

# Join target from 1 future month (not seen at training or prediction time)
future_target = data[(data['model'].isin(train_models)) & (data['start']==test_start)][cat_cols+['channel_sales_estimate']]
future_target = future_target.rename(columns={'channel_sales_estimate':'future_sales_estimate'})
data_test = data_test.merge(future_target,on=cat_cols)

In [13]:
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoded_columns = [item+'_enc' for item in cat_cols]
data_train_enc = data_train[all_cols].join(pd.DataFrame(enc.fit_transform(data_train[cat_cols]),columns=encoded_columns).astype(int)).fillna(0)
data_test_enc = data_test[all_cols].reset_index(drop=True).join(pd.DataFrame(enc.transform(data_test[cat_cols]),columns=encoded_columns).astype(int)).fillna(0)

In [14]:
def df_to_json(data_with_encoding):
    data_modeling = data_with_encoding.copy()
    data_modeling['target'] = data_modeling[target_col]
    data_modeling['dynamic_feat'] = data_modeling[dynamic_cols].values.tolist()
    data_modeling['cat'] = data_modeling[encoded_columns].values.tolist()
    return data_modeling[['start','target','dynamic_feat','cat']].to_json(orient='records', lines=True)

In [15]:
training_data = df_to_json(data_train_enc)
test_data = df_to_json(data_test_enc)

In [16]:
def write_dicts_to_file(path, data):
    with open(path, "wb") as fp:
        fp.write(data.encode("utf-8"))

In [17]:
%%time
write_dicts_to_file("train.json", training_data)
write_dicts_to_file("test.json", test_data)

CPU times: user 8.3 ms, sys: 16.1 ms, total: 24.4 ms
Wall time: 249 ms


In [18]:
s3 = boto3.resource("s3")


def copy_to_s3(local_file, s3_path, override=False):
    assert s3_path.startswith("s3://")
    split = s3_path.split("/")
    bucket = split[2]
    path = "/".join(split[3:])
    buk = s3.Bucket(bucket)

    if len(list(buk.objects.filter(Prefix=path))) > 0:
        if not override:
            print(
                "File s3://{}/{} already exists.\nSet override to upload anyway.\n".format(
                    s3_bucket, s3_path
                )
            )
            return
        else:
            print("Overwriting existing file")
    with open(local_file, "rb") as data:
        print("Uploading file to {}".format(s3_path))
        buk.put_object(Key=path, Body=data)

In [19]:
%%time
copy_to_s3("train.json", s3_output_path + "/train/train.json",override=True)
copy_to_s3("test.json", s3_output_path + "/test/test.json",override=True)

Uploading file to s3://tdsp-projects-dev/mds/sales_incentive/sandbox/ankit/forecast_output/20241210070332/train/train.json
Uploading file to s3://tdsp-projects-dev/mds/sales_incentive/sandbox/ankit/forecast_output/20241210070332/test/test.json
CPU times: user 78.6 ms, sys: 44.2 ms, total: 123 ms
Wall time: 529 ms


In [20]:
estimator = sagemaker.estimator.Estimator(
    image_uri=image_name,
    sagemaker_session=sagemaker_session,
    role=role,
    train_instance_count=1,
    train_instance_type="ml.c4.2xlarge",
    base_job_name="deepar-incentives",
    output_path=s3_output_path,
)

train_instance_count has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.
train_instance_type has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


In [21]:
freq='1M'
forecast_horizon=1
hyperparameters = {
    "time_freq": freq,
    "epochs": "50",
    "early_stopping_patience": "40",
    "mini_batch_size": "64",
    "learning_rate": "5E-4",
    "context_length": str(forecast_horizon),
    "prediction_length": str(forecast_horizon),
}

In [22]:
estimator.set_hyperparameters(**hyperparameters)

In [23]:
%%time
data_channels = {"train": "{}/train/".format(s3_output_path)}

estimator.fit(inputs=data_channels, wait=True)

INFO:sagemaker:Creating training-job with name: deepar-incentives-2025-02-18-14-09-40-864


2025-02-18 14:09:42 Starting - Starting the training job...
2025-02-18 14:09:56 Starting - Preparing the instances for training...
2025-02-18 14:10:35 Downloading - Downloading the training image.........
2025-02-18 14:12:12 Training - Training image download completed. Training in progress...Docker entrypoint called with argument(s): train
Running default environment configuration script
Running custom environment configuration script
/opt/amazon/lib/python3.8/site-packages/mxnet/model.py:97: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if num_device is 1 and 'dist' not in kvstore:
[02/18/2025 14:12:28 INFO 139965563537216] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'_kvstore': 'auto', '_num_gpus': 'auto', '_num_kv_servers': 'auto', '_tuning_objective_metric': '', 'cardinality': 'auto', 'dropout_rate': '0.10', 'early_stopping_patience': '', 'embedding_dimension': '10', 'learning_rate': '0.001', 'likel

In [ ]:
batch_input = s3_output_path + "/test/test.json"
batch_output = 's3://{}/{}/batch-inference'.format(s3_bucket, s3_output_prefix)

transformer = estimator.transformer(instance_count=1, instance_type='ml.m4.xlarge', output_path=batch_output,assemble_with='Line')

transformer.transform(data=batch_input, data_type='S3Prefix', split_type='Line')

transformer.wait()

INFO:sagemaker:Creating model with name: deepar-incentives-2025-02-18-14-15-43-808
INFO:sagemaker:Creating transform job with name: deepar-incentives-2025-02-18-14-15-44-610


...................

In [ ]:
predictions = pd.read_json(batch_output+ "/test.json.out",orient='records',lines=True,encoding='utf-8')
predictions['mean'] = predictions['mean'].apply(lambda x: x[0])
predictions = predictions.rename(columns={'mean':'future_prediction'})

In [ ]:
test_pred = data_test.join(predictions)

In [ ]:
plt.scatter(test_pred['future_prediction'],test_pred['future_sales_estimate'])
plt.xlabel('Predicted Sales')
plt.ylabel('Actual Sales')

In [ ]:
test_pred.groupby(['toyota_region_area','model','current_model_year','type_of_sale'])[['future_sales_estimate','future_prediction']].sum().reset_index().head(40)

In [ ]:
evaluation = test_pred[(test_pred['model'].isin(['Camry Hybrid','Corolla','Highlander','RAV4','Tacoma','Tundra']))].groupby(['model'])[['future_sales_estimate','future_prediction']].sum()
evaluation